In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/danielihenacho/amazon-reviews-dataset/cleaned_reviews.csv


In [17]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

import re
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
df = pd.read_csv('/kaggle/input/datasets/danielihenacho/amazon-reviews-dataset/cleaned_reviews.csv')
df.head()

,sentiments,cleaned_review,cleaned_review_length,review_score
0,positive,i wish would have gotten one earlier love it a...,19,5.0
1,neutral,i ve learned this lesson again open the packag...,88,1.0
2,neutral,it is so slow and lags find better option,9,2.0
3,neutral,roller ball stopped working within months of m...,12,1.0
4,neutral,i like the color and size but it few days out ...,21,1.0


In [19]:
X = df['cleaned_review']
y = df['sentiments']

In [20]:
le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

['negative' 'neutral' 'positive']


In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
X_train = X_train.astype(str)
X_test = X_test.astype(str)

In [23]:
max_vocab = 10000
max_len = 100

vectorizer = layers.TextVectorization(
    max_tokens=max_vocab,
    output_mode='int',
    output_sequence_length=max_len
)

vectorizer.adapt(X_train)

In [24]:
X_train_vec = vectorizer(X_train)
X_test_vec = vectorizer(X_test)

In [25]:
model = keras.Sequential([
    layers.Embedding(input_dim=max_vocab, output_dim=128, input_length=max_len),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [26]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [27]:
history = model.fit(
    X_train_vec,
    y_train,
    validation_data=(X_test_vec, y_test),
    epochs=5,
    batch_size=32
)

Epoch 1/5
434/434 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5694 - loss: 0.8809 - val_accuracy: 0.7592 - val_loss: 0.6306
Epoch 2/5
434/434 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.7479 - loss: 0.5963 - val_accuracy: 0.7659 - val_loss: 0.5334
Epoch 3/5
434/434 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8025 - loss: 0.4863 - val_accuracy: 0.8181 - val_loss: 0.4650
Epoch 4/5
434/434 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8366 - loss: 0.4172 - val_accuracy: 0.8209 - val_loss: 0.4517
Epoch 5/5
434/434 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8478 - loss: 0.3827 - val_accuracy: 0.8547 - val_loss: 0.3979


# Pytorch

In [28]:
import pandas as pd
import numpy as np
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from collections import Counter

In [29]:
df = pd.read_csv('/kaggle/input/datasets/danielihenacho/amazon-reviews-dataset/cleaned_reviews.csv')
df = df.dropna(subset=['cleaned_review'])

X = df['cleaned_review'].astype(str)
y = df['sentiments']

le = LabelEncoder()
y = le.fit_transform(y)

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [31]:
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

vocab = Counter()
for text in X_train:
    vocab.update(tokenize(text))

vocab = {word: i+2 for i, (word, _) in enumerate(vocab.most_common(10000))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

In [32]:
max_len = 100

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(t, 1) for t in tokens]
    ids = ids[:max_len]
    ids += [0] * (max_len - len(ids))
    return ids

X_train_enc = [encode(t) for t in X_train]
X_test_enc = [encode(t) for t in X_test]

In [33]:
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TextDataset(X_train_enc, y_train)
test_dataset = TextDataset(X_test_enc, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [34]:
class TextModel(nn.Module):
    def __init__(self, vocab_size):
        super(TextModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, 128, padding_idx=0)
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.embedding(x)
        x = x.mean(dim=1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = TextModel(len(vocab))

In [35]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [36]:
epochs = 5
for epoch in range(epochs):
    model.train()
    for xb, yb in train_loader:
        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.6874
Epoch 2, Loss: 0.3557
Epoch 3, Loss: 0.1560
Epoch 4, Loss: 0.7678
Epoch 5, Loss: 0.1927


In [37]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb)
        preds = torch.argmax(preds, dim=1)

        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds, average='weighted'))

Accuracy: 0.8526528258362168
F1: 0.8448300509300807
